#### LLM大模型连接与通信

In [4]:
import os
import dotenv
dotenv.load_dotenv()
print(os.getenv("openai_base_url_chat"))
print(os.getenv("openai_api_key_chat"))

https://api.chatanywhere.tech/v1
sk-VFxxxxxxxxxxxxxxxxhKd016


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# 1. 初始化连接（配置 API Key 和 模型名称）
# 如果换用 Anthropic，只需 from langchain_anthropic import ChatAnthropic
llm = ChatOpenAI(
    temperature=0.7,
    model="gpt-3.5-turbo",
    api_key=os.getenv("openai_api_key_chat"),
    base_url=os.getenv("openai_base_url_chat"),
    timeout=30,           # 通信超时时间
    max_retries=2         # 失败重试次数
)

# 2. 标准消息格式通信
messages = [
    SystemMessage(content="你是一个资深Python架构师，回答要简洁专业。"),
    HumanMessage(content="解释一下 @classmethod 的作用。")
]

# 3. 同步调用
# response = llm.invoke(messages)
# print(response.content)

# 4. 流式通信（推荐用于前端展示，避免长时间白屏等待）
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)


PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


`@classmethod` 是 Python 中的一种装饰器，用于定义类方法。与实例方法不同，类方法的第一个参数是 `cls`，而不是 `self`。这意味着类方法可以通过类本身调用，而不需要实例化对象。

主要作用包括：

1. **访问类属性**：类方法能够访问和修改类属性，而实例方法只能访问实例属性。
  
2. **工厂方法**：可以作为创建类实例的替代构造器。

3. **继承支持**：类方法可以在子类中被重写，能够利用多态性。

示例代码：

```python
class MyClass:
    count = 0

    def __init__(self):
        MyClass.count += 1

    @classmethod
    def get_count(cls):
        return cls.count

# 使用类方法
print(MyClass.get_count())  # 输出 0
obj1 = MyClass()
print(MyClass.get_count())  # 输出 1
``` 

通过这个示例，可以看到 `get_count` 方法如何通过类本身访问 `count` 属性。

## LangChain 四个核心动词：invoke / stream / astream / batch

假设我们向模型提问：**“写一篇500字的作文”**。

- **`invoke`**：同步、阻塞；**一次性**返回完整结果。
- **`stream`**：同步、阻塞；**分块（chunk）**返回，适合边生成边展示。
- **`astream`**：异步、非阻塞；**分块（chunk）**返回，适合异步服务/并发。
- **`batch`**：同步、阻塞；**一次性**返回多个输入的结果（批量）。

下面给出四种代码示例（默认复用你上面已经创建好的 `llm`）。

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

# 统一的提问
essay_messages = [
    SystemMessage(content="你是一位中文写作老师，文章结构清晰，语言自然。"),
    HumanMessage(content="写一篇500字的作文，题目《春天的校园》。")
]

In [7]:
# 1) invoke：同步 + 一次性返回完整结果
resp = llm.invoke(essay_messages)
print(resp.content)

**春天的校园**

春天，万物复苏，校园里的一切都焕发出新的生机。走进校园，映入眼帘的是那片嫩绿的新芽，仿佛是大自然为校园披上的一层轻纱。树枝上新长出的叶子在阳光的照耀下，透出迷人的光泽，诉说着春天的故事。

校园的小路两旁，花儿竞相开放。五颜六色的花朵经过寒冬的沉睡，争先恐后地绽放，红的、黄的、紫的，如同小女孩的裙子，随风摇曳。尤其是那一丛丛的樱花，粉色的花瓣在春风中轻轻舞动，仿佛在为我们演绎一场美丽的舞蹈。每当这个时候，我总爱和同学们在樱花树下驻足，享受这份美好。花瓣纷纷扬扬从树上飘落，像是春天的雪花，带来满园的芬芳与诗意。

校园的操场上，孩子们的欢声笑语似乎把整个春天都唤醒了。男生们在追逐打闹，女生们则在欢快地跳绳，笑声在空中回荡，仿佛是春天的乐章。在他们的笑脸上，我看到了一种无忧无虑的生活态度，感受到了春天的温暖和希望。这样的场景让我想起了自己童年时的春天，那时的我也是如此天真快乐，爱在花海中奔跑，心中充满了对未来的憧憬。

除了花草树木，春天的校园还有着独特的文化气息。伴随着温暖的阳光，老师们在课堂上讲解知识的过程如同春风化雨，滋润着我们求知的心田。此时，课本中枯燥的文字仿佛也变得生动起来，历史的故事、科学的奥秘在这样的春日里都有了更加鲜活的表达。我们在知识的海洋里遨游，感受着春天赋予我们的智慧。

放学后，校园的小径上常常能见到学生们在讨论学习、交流心得。春天的校园不仅是学习的摇篮，更是友谊的温床。在这个充满阳光和暖风的季节里，我们的心灵因友谊的滋养而蓬勃发展。

总之，春天的校园如同一幅绚丽的画卷，勾勒出生命的活力与希望。在这片充满生机的土地上，我们共同成长，书写着属于我们的青春篇章。春天的校园，不仅展现了大自然的美丽，更寄托着我们心中对未来的美好向往。


In [8]:
# 2) stream：同步 + 分块返回（边生成边输出）
for chunk in llm.stream(essay_messages):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()  # 换行

《春天的校园》

春天，万物复苏，校园里的每一个角落似乎都在欢呼、歌唱。清晨，温暖的阳光透过树梢，洒在校园的操场上，洒在那片柔软的草地上，给人一种无比舒适的感觉。学生们纷纷走出教室，迎接这个充满生机的季节。

走进校园，首先映入眼帘的是那一排排盛开的樱花树。粉红色的花瓣在微风中摇曳，犹如仙女洒下的花雨。每当课间，许多同学们聚在樱花树下，嬉笑打闹，纷纷用手机记录下这美丽的瞬间。樱花的芬芳弥漫在空气中，带着春天的气息，让人心旷神怡。树上飞来的小鸟，叽叽喳喳地唱着春天的歌，增添了校园的活力。

走过操场，草坪上又是一片热闹的景象。阳光照耀下，孩子们在草地上追逐嬉戏，有的在踢足球，有的在打羽毛球，还有的在草地上铺开一本课外书，静静地阅读。春天的校园，仿佛是大自然给予孩子们的一个舞台，尽情展现着他们的朝气与活力。此时，可以看到一队队的孩子，满脸的快乐在晨光中流露，笑声传遍整个校园。

校园里的教学楼也被春天的气息陶醉了。窗外，阳光透过绿叶洒在白墙上，映出斑驳的光影。教室里，老师在板书时，窗外的春色吸引了学生们的视线。每当教师提问时，总能看到孩子们饱满的求知欲，仿佛他们的心灵也在春风的吹拂下悸动着。课堂上的氛围因春天的到来而愈加活跃，大家纷纷踊跃发言，表达对知识的渴求。

春天的校园，是希望与梦想交汇的地方。这里不仅孕育着知识的种子，也让孩子们在阳光下茁壮成长。每一个角落、每一片花草都在诉说着春天的故事，讲述着我们的青春。无论时光如何流转，这样的春天都将深深铭刻在每个学子的心中，成为他们人生旅途中的一抹亮色。

春天的校园如此美丽，如此生机勃勃。它不仅是学习的乐园，更是我们心灵的归宿。在这里，我们欢笑、成长，播种希望，迎接未来。


In [11]:
# 3) astream：异步 + 分块返回（适合并发/异步服务）
# 说明：有些版本/模型返回的 chunk 里，content 可能阶段性为空，
#      因此不要用 `if chunk.content:` 过早过滤；先尽量打印/累积。

async def run_astream():
    parts = []
    async for chunk in llm.astream(essay_messages):
        content = getattr(chunk, "content", None)

        # content 通常是 str；少数情况下可能是 list/None
        if isinstance(content, str):
            if content:
                print(content, end="", flush=True)
            parts.append(content)
        elif content is not None:
            # 兜底：直接打印非字符串内容，避免“看起来没输出”
            print(str(content), end="", flush=True)
            parts.append(str(content))
        else:
            # 再兜底：把 chunk 的结构打出来，方便定位
            # （只打很短的一行，避免刷屏）
            parts.append("")

    final_text = "".join([p for p in parts if isinstance(p, str)])
    if final_text.strip() == "":
        print("\n[debug] astream 收到的 chunk.content 全为空；请改用下面的 ainvoke/stream 或检查依赖版本。")
    else:
        print()  # 换行

await run_astream()

《春天的校园》

春天，是一个充满生机和希望的季节。在这个温暖的季节里，我的校园也如同一幅美丽的画卷，展现出自然的魅力和生命的活力。

走进校园，首先映入眼帘的是那一片片绿油油的草地。经过冬季的沉寂，草地上的小草们悄悄地探出了头，嫩绿的颜色犹如新生的生命，散发着清新的气息。课间，同学们在草地上嬉戏打闹，欢声笑语此起彼伏，犹如春天的乐章，在校园中荡漾开来。

校园的花坛里，各种花朵纷纷绽放，形成了一片五彩斑斓的海洋。桃花、樱花、迎春花交相辉映，香气扑鼻而来，吸引着蜜蜂和蝶子的到访。桃花粉红如霞，樱花洁白如雪，迎春花则如晨曦一般灿烂。这些花儿不仅装点了校园的美丽，也让我们的心情在春日的阳光中暖洋洋地被唤醒。

在这个季节里，校园里的大树也焕发出新的活力。高大的榕树和挺拔的柏树，经过寒冬的洗礼，开始抽出嫩芽，嫩绿的叶子在春风的吹拂下轻轻摇曳，宛如在向我们招手。树下是我们学习和交流的场所，课外活动时，同学们在树荫下聚集在一起，享受着春天的美好时光。

春天的校园，还有那如歌的鸟儿。清晨，清脆的鸟鸣声伴随着微风钻入耳中，仿佛是在告诉我们：春天来了！小鸟们在校园的枝头欢快地歌唱，宛如自然界的歌手，让整个校园充满了生机与活力。在这样的环境中，心中所有的烦恼似乎都被驱散，留下的只有宁静与快乐。

春天的校园，不仅是自然的变化，更是我们心灵的慰藉。看着校园里的一切，我明白了春天不仅仅是一个季节，更是一种希望，一种新的开始。它提醒我们，无论生活多么艰难，总会有阳光洒进来，让我们重新绽放。让我们珍惜这美好的春天，在校园中播撒下自己的希望，让未来更加美好。


In [13]:
# 4) batch：同步 + 批量输入，一次性返回多个结果
batch_inputs = [
    essay_messages,
    [
        SystemMessage(content="你是一位中文写作老师，文章结构清晰，语言自然。"),
        HumanMessage(content="写一篇500字的作文，题目《雨后的城市》。")
    ],
]

batch_resps = llm.batch(batch_inputs)
for i, r in enumerate(batch_resps, start=1):
    print(f"\n--- 第{i}篇 ---")
    print(r.content)


--- 第1篇 ---
《春天的校园》

春天，万物复苏，校园里仿佛也披上了一层生机勃勃的绿衣。走进校园，迎面而来的是那扑鼻而来的花香，令人陶醉。柳树吐出了嫩芽，轻轻飘舞的柳条在微风中摇曳，仿佛在向我们招手，呼唤着春天的到来。

校园的花坛中，各色花朵争相开放。樱花如云霞般绚烂，娇艳欲滴，吸引了无数的蜜蜂和蝴蝶翩翩而至。它们在花间穿梭，表现出春天的活泼与灵动。郁金香则高傲而优雅，挺立在花坛中，仿佛在宣告自己的美丽。而那些五颜六色的迎春花如同星星点点，为校园增添了无尽的生机。

在这个季节，学生们的心情也随着春天的到来而变得愉悦。早晨，操场上传来了孩子们的欢声笑语，他们在阳光下尽情地奔跑。女生们在树下搭起了一个小摊，售卖自制的点心，男生们则在一旁打篮球，汗水浸透了他们的T恤，却丝毫不影响他们的兴致。每一个角落都洋溢着青春的活力，校园充满了轻松愉快的氛围。

教室里，老师们在课堂上讲述着春天的知识，激发着学生们对自然的热爱。窗外，阳光透过树叶洒进教室，形成斑驳的光影，学生们的脸上洋溢着求知的渴望。课间，大家三三两两地聚在一起，讨论着春天的趣事，描绘着未来的梦想。这样的校园生活，仿佛为我们的学习注入了新的动力。

而在图书馆中，春天的气息同样弥漫。许多同学趁着春光明媚，纷纷来到这里，捧着书本，沉浸在知识的海洋中。阳光透过窗户洒在书页上，映出一片温暖的金色。每个人都在为了自己的梦想而努力，仿佛这一切都在预示着新生与希望。

春天的校园，犹如一幅美丽的画卷，充满了生机与活力。它不仅是学习的殿堂，更是我们心灵的家园。让我们珍惜这美好的春天，携手共进，迎接未来的每一个挑战。

--- 第2篇 ---
题目：《雨后的城市》

一场春雨过后，城市的面貌仿佛焕然一新。阳光透过云层，洒下温暖的余辉，湿润的空气中弥漫着泥土和青草的清香，整个城市散发着一种生机勃勃的气息。

走出家门，首先迎面而来的便是路边那一层闪闪发光的水珠。树叶上的雨滴在阳光的照射下，如同钻石般璀璨，犹如精灵在舞蹈。行人们的脸上都挂着满足的微笑，仿佛这一场雨不仅洗净了城市的尘埃，也洗涤了心灵的烦躁。

漫步在街道上，路面上积水的小水洼映出蓝天白云的倒影，宛若一幅流动的画卷。小朋友们兴高采烈地跳跃在水洼旁，欢乐的笑声回荡在空气中。路边的花坛里，经过雨水的滋润，各色花朵争相开放，鲜艳的色彩为城市增添了无限的生机。

走到公